<div style="background: linear-gradient(135deg, rgb(196, 196, 196) 0%, #5a5a5aeb 50%, rgb(0, 0, 0) 100%);
            padding: 10px; border-radius: 8px; margin: 10px 0; color: black; font-family: Arial, sans-serif; text-align: center; font-size: 24px;">

# **Тестирование и сравнения моделей**
</div>

Ноутбук предназначен для загрузки обученных моделей, их тестирования на различных примерах и сравнения качества генерации между LSTM и трансформерной (DistilGPT2) архитектурами.

Основная цель: Провести всестороннее тестирование обученной LSTM модели, сравнить её с предобученной моделью DistilGPT2 и визуализировать результаты генерации для качественной и количественной оценки.

Основные задачи:
1. Загрузка и подготовка окружения, загрузка данных
1. Загрузка моделей
1. Тестирование и анализ моделей
1. Количественная оценка и визуализация результатов

In [1]:
# Импорт основных библиотек и модулей
import sys
import os
sys.path.append('..')

import torch
import pandas as pd
from transformers import GPT2TokenizerFast

from src.data_utils import load_split_data, load_config
from src.next_token_dataset import get_tokenizer, NextTokenDataset
from src.lstm_model import LSTMLanguageModel
from src.eval_lstm import test_lstm_examples, evaluate_rouge_lstm
from src.eval_transformer_pipeline import TransformerEvaluator

In [2]:
# Настройка устройства
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    torch.cuda.empty_cache()

Device: cuda
GPU: NVIDIA GeForce RTX 4070


In [3]:
# Загрузка конфигурации
config = load_config("configs/config.yaml")

Конфигурация загружена configs/config.yaml


In [ ]:
# Загрузка данных
train_texts, val_texts, test_texts = load_split_data()



Загружены разделенные датасеты:
Train: 800000
Validation: 100000
Test: 100000
Train: 800000
Validation: 100000
Test: 100000


In [5]:
# Загрузка токенизатора
tokenizer = get_tokenizer(config['tokenization']['model_name'])
pad_id = tokenizer.pad_token_id

Проведем повторно:
- инициализацию архитектуры LSTM с сохраненными параметрами
- загрузку весов из `models/best_lstm_model.pt`
- визуализацию примеров LSTM

In [6]:
# Загрузка обученной LSTM модели
vocab_size = tokenizer.vocab_size

lstm_model = LSTMLanguageModel(
    vocab_size=vocab_size,
    pad_id=pad_id,
    embed_dim=config['lstm']['embed_dim'],
    hidden_dim=config['lstm']['hidden_dim'],
    num_layers=config['lstm']['num_layers'],
    dropout=config['lstm']['dropout']
).to(device)

In [7]:
# Загружаем веса
if os.path.exists('models/best_lstm_model.pt'):
    lstm_model.load_state_dict(torch.load('models/best_lstm_model.pt', 
                                          map_location=device))
    lstm_model.eval()
    print("LSTM модель загружена")
else:
    print("Модель не найдена!")

LSTM модель загружена


C:\Users\xndrf\AppData\Local\Temp\ipykernel_13436\3400389791.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  lstm_model.load_state_dict(torch.load('models/best_lstm_mode

In [18]:
# Тестирование LSTM на примерах
test_lstm_examples(lstm_model, tokenizer, device)


Промпт: Hello, my name is
Генерация: Hello, my name is, i'm a sucker for a baby. i can't go to the

Промпт: Today is a beautiful
Генерация: Today is a beautiful idea! i'm looking forward to it! how long is it? i

Промпт: How are you doing
Генерация: How are you doing that? im not a fan of the song, but i'm too shy

Промпт: I love to
Генерация: I love to the best way to spend the day with the family, and happy mother's

Промпт: Machine learning is
Генерация: Machine learning is a nice job. i love the music. the same thing. you're


In [16]:
# Оценка LSTM на тестовой выборке
from torch.utils.data import DataLoader
from src.next_token_dataset import collate_fn

# Создаем тестовый датасет и загрузчик
test_dataset = NextTokenDataset(test_texts, tokenizer, config['tokenization']['max_len'])
test_loader = DataLoader(
    test_dataset,
    batch_size=config['training']['batch_size'],
    collate_fn=lambda b: collate_fn(b, pad_id)
)

# Оценка ROUGE
rouge1, rouge2 = evaluate_rouge_lstm(
    lstm_model, test_loader, tokenizer, device, 
    num_examples=config['evaluation']['rouge_examples']
)

print(f"\nLSTM на тестовой выборке:")
print(f"ROUGE-1: {rouge1:.4f}")
print(f"ROUGE-2: {rouge2:.4f}")


LSTM на тестовой выборке:
ROUGE-1: 0.0845
ROUGE-2: 0.0209


Ну собственно говоря ничего нового, как и в `notebook_LSTM.ipynb`

Для `distilgpt2` проведем эксперименты с параметрами генерации различных **temperature**:
- 0.5 (консервативная генерация)
- 0.7 (сбалансированная)
- 0.9 (креативная)
- 1.2 (очень креативная)


In [23]:
# Инициализация трансформера
transformer_eval = TransformerEvaluator("distilgpt2")

# Тестирование параметров трансформера
transformer_eval.test_parameters("Hello, my name is", max_new_tokens=15)

Device set to use cuda:0



Промпт: 'Hello, my name is'

Консервативный:
  Hello, my name is Tanya.

I am a small girl with a very short hair

Сбалансированный:
  Hello, my name is Dan. I’m a fan of the anime, and I�

Креативный:
  Hello, my name is Richard.
As I said I was very lucky to have such an amazing

Очень креативный:
  Hello, my name is Liza, a Chinese Chinese professional who took part in the GK (


Можно сказать, что консервативный очень простой и приближенный к реальности, Сбалансированные метрики - уже как-то связные и релевантные дает ответы, с увеличением креатива смысл теряется становится текст хаотичным.

In [13]:
# Оценка трансформера на тестовой выборке
gpt2_rouge1, gpt2_rouge2 = transformer_eval.evaluate_rouge(
    test_texts, 
    num_examples=config['evaluation']['rouge_examples'],
    split_ratio=0.75
)

print(f"\nDistilGPT2 на тестовой выборке:")
print(f"ROUGE-1: {gpt2_rouge1:.4f}")
print(f"ROUGE-2: {gpt2_rouge2:.4f}")

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset



DistilGPT2 на тестовой выборке:
ROUGE-1: 0.0664
ROUGE-2: 0.0000


In [ ]:
# Сравнение моделей
print(f"{'Модель':<20} {'ROUGE-1':<10} {'ROUGE-2':<10}")
print(f"{'LSTM':<20} {rouge1:<10.4f} {rouge2:<10.4f}")
print(f"{'DistilGPT2':<20} {gpt2_rouge1:<10.4f} {gpt2_rouge2:<10.4f}")

# Примеры генерации
examples = [
    "My name is",
    "Good morning, today I want to talk about",
    "What is your favorite",
    "In my opinion",
    "Yandex is a company that"
]

for prompt in examples:
    print(f"\nПромпт: '{prompt}'")
    
    # LSTM
    lstm_output = lstm_model.generate(
        tokenizer,
        prompt,
        max_length=15,
        device=device,
        temperature=0.7,
        top_k=50,
        top_p=0.9
    )
    print(f"LSTM: {lstm_output}")
    
    # GPT2
    gpt2_output = transformer_eval.generate(
        prompt,
        max_new_tokens=15,
        temperature=0.7,
        top_k=50,
        top_p=0.9
    )
    print(f"GPT2: {gpt2_output}")


Модель               ROUGE-1    ROUGE-2   
LSTM                 0.0845     0.0209    
DistilGPT2           0.0664     0.0000    

Промпт: 'My name is'
LSTM: My name is a beautiful. i'm not going to be able to watch the show anymore
GPT2: My name is, in the world of the internet, all the time.”


Промпт: 'Good morning, today I want to talk about'
LSTM: Good morning, today I want to talk about my friend and i miss him. i want him to be in a coma
GPT2: Good morning, today I want to talk about the topic of the U.S. Postal Service. It‍s

Промпт: 'What is your favorite'
LSTM: What is your favorite thing i am in the same place! it's so much fun, i
GPT2: What is your favorite TV show?













Промпт: 'In my opinion'
LSTM: In my opinion on the day! how are you? have a good day! xx,
GPT2: In my opinion, there is a lot of reason to believe that I'm not a great

Промпт: 'Yandex is a company that'
LSTM: Yandex is a company that i don't have a friend i want to be in my friends house in
GPT2: Ya

<div style="font-family: 'Courier New', monospace; font-size: 120%; background: linear-gradient(135deg, rgb(126, 126, 126) 0%, #f4f4f4 30%, #ffffff 60%, #000000 100%); border-left: 4px solid #ffffff; padding: 12px; margin: 10px 0; border-radius: 8px; color: rgb(0, 0, 0); box-shadow: 0 0 15px rgba(71, 255, 237, 0.6), 0 0 20px rgba(255, 71, 87, 0.2); text-shadow: 0 1px 1px rgba(0, 0, 0, 0.4);">
<strong style="color: rgb(0, 0, 0); font-weight: 600;">Результаты тестирования</strong>
</div>

Сравнение ROUGE:
- ROUGE-1: 0.0845 vs 0.0664
- ROUGE-2: 0.0209 vs 0.0000

Можно сказать, что LSTM чаще совпадает по словам и биграммам с эталоном.

Однако значения очень низкие у обеих моделей. Почему так?
Мои предположения в том, качество данных хромает, необходимо обучение на лучших источниках, а потом уже непосредственно доработка стилистики согласно твитам.

Касательно качества текста, то LSTM - часто теряет смысл и дает нелогичные фразы с повторениями, а distilgpt2 дает более естественную и связную речь похожую на реальную.

Если мы упираемся в метрики, то лучше LSTM, если важен качество текстов то gpt2, а по хорошему выполнить дообучение на дополнительных данных